# EDA of parents in anatomy dataset

In [1]:
# ruff: noqa: T201, T203
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging

import pandas as pd
from dotenv import load_dotenv

from config.config import DATA_DIR
from src.evaluate import analyze_results, get_predictions_with_gt, plot_usage_histograms
from src.formatting import (
    format_oracle_pairs_filepath,
    format_run_path,
    format_storing_pathes_from_run_path,
)
from src.LLM_servers.openai import OpenAIServer
from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.processing import parallel_samples_process
from src.prompts.system import BASELINE_INITIALIZATION_MESSAGE
from src.utils import read_oracle_pairs, save_run_results

logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

In [2]:
llm_oracle = OpenAIServer()
llm_oracle.add_system_context(BASELINE_INITIALIZATION_MESSAGE)

In [3]:
DATASET_NAME = "anatomy"
SET_NAME = "human-mouse"
SOURCE_ONTOLOGY = "mouse"
TARGET_ONTOLOGY = "human"

src_onto_path = DATA_DIR / DATASET_NAME / SET_NAME / f"{SOURCE_ONTOLOGY}.owl"
tgt_onto_path = DATA_DIR / DATASET_NAME / SET_NAME / f"{TARGET_ONTOLOGY}.owl"

onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)

* Owlready2 * Creating new ontology human <data/anatomy/human-mouse/human.owl#>.
* Owlready2 * ADD TRIPLE data/anatomy/human-mouse/human.owl http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Ontology
* Owlready2 *     ...loading ontology human from data/anatomy/human-mouse/human.owl...


* Owlready2 * Reseting property oboInOwl.ObsoleteProperty: new triples are now available.
* Owlready2 * Reseting property oboInOwl.hasRelatedSynonym: new triples are now available.
* Owlready2 * Reseting property oboInOwl.hasDefaultNamespace: new triples are now available.
* Owlready2 * Reseting property oboInOwl.savedBy: new triples are now available.
* Owlready2 * Reseting property oboInOwl.hasDate: new triples are now available.


* Owlready2 *     ...8 properties found: ObsoleteProperty, UNDEFINED_part_of, label, hasRelatedSynonym, hasDefaultNamespace, savedBy, hasDate, hasDefinition


In [81]:
candidate_pairs_lines = pd.DataFrame(
    read_oracle_pairs(format_oracle_pairs_filepath(DATASET_NAME, SET_NAME)), columns=["source", "target"]
)

In [71]:
candidate_pairs_lines.head()

,source,target
0,http://human.owl#NCI_C32658,http://mouse.owl#MA_0002684
1,http://human.owl#NCI_C33075,http://mouse.owl#MA_0001249
2,http://human.owl#NCI_C33748,http://mouse.owl#MA_0001221
3,http://human.owl#NCI_C12262,http://mouse.owl#MA_0001615
4,http://human.owl#NCI_C53050,http://mouse.owl#MA_0002234


In [ ]:
for col in candidate_pairs_lines.columns:
    print(f"# Total elements {col}: {candidate_pairs_lines[col].shape[0]}")
    print(f"# Duplicates {col}: {candidate_pairs_lines[col].duplicated().sum()}")
    print(f"# Unique elements {col}: {candidate_pairs_lines[col].nunique()}")

# Total elements source: 398
# Duplicates source: 25
# Unique elements source: 373
# Total elements target: 398
# Duplicates target: 20
# Unique elements target: 378


### Restructure source - target ontologies

In [13]:
target_cols = candidate_pairs_lines[0]
source_cols = candidate_pairs_lines[1]

In [84]:
def get_df(cols: list[str], onto: OntologyAccess) -> pd.DataFrame:
    data = [
        {
            "entry": col,
            "all_entity_names": OntologyEntryAttr(col, onto).get_all_entity_names(),
            "parents_preferred_names": OntologyEntryAttr(col, onto).get_parents_preferred_names(),
            "direct_parents": OntologyEntryAttr(col, onto).get_direct_parents(),
        }
        for col in cols
    ]
    return pd.DataFrame(data)

tgt_df = get_df(target_cols, onto_tgt)
src_df = get_df(source_cols, onto_src)

In [ ]:
src_df["parents_preferred_names"] = src_df["parents_preferred_names"].apply(lambda x: list(map(str, x)))
tgt_df["parents_preferred_names"] = tgt_df["parents_preferred_names"].apply(lambda x: list(map(str, x)))

src_df["direct_parents"] = src_df["direct_parents"].astype(str)
tgt_df["direct_parents"] = tgt_df["direct_parents"].astype(str)

In [86]:
src_df.head()

,entry,all_entity_names,parents_preferred_names,direct_parents
0,http://mouse.owl#MA_0002684,{stomach muscularis mucosa},[{'Thing'}],{owl.Thing}
1,http://mouse.owl#MA_0001249,"{palpebral gland, tarsal gland, meibomian gland}","[{'eye gland'}, {'Thing'}]",{mouse.MA_0000267}
2,http://mouse.owl#MA_0001221,{tensor tympani},"[{'tympanic cavity muscle'}, {'Thing'}]",{mouse.MA_0000256}
3,http://mouse.owl#MA_0001615,{stomach greater curvature},"[{'Thing'}, {'stomach region'}]",{mouse.MA_0002561}
4,http://mouse.owl#MA_0002234,{testicular vein},"[{'vein'}, {'venous blood vessel'}, {'blood ve...",{mouse.MA_0000067}


In [83]:
tgt_df.head()

,entry,all_entity_names,parents_preferred_names,direct_parents
0,http://human.owl#NCI_C32658,"{Gastric Muscularis Mucosa, Gastric_Muscularis...","{'Mucosa'}, {'Thing'}, {'Anatomic_Structure_Sy...","{human.NCI_C33149, human.NCI_C32656}"
1,http://human.owl#NCI_C33075,"{Meibomian Gland, Meibomian_Gland}","{'Sebaceous_Gland'}, {'Thing'}, {'Anatomic_Str...",{human.NCI_C33519}
2,http://human.owl#NCI_C33748,"{Tensor Tympani, Tensor_Tympani}","{'Body_Part'}, {'Thing'}, {'Anatomic_Structure...",{human.NCI_C33148}
3,http://human.owl#NCI_C12262,"{Greater Curvature, Greater Curvature of the S...","{'Anatomic_Structure_System_or_Substance'}, {'...",{human.NCI_C25763}
4,http://human.owl#NCI_C53050,"{Testicular Vein, Spermatic Vein, Internal Spe...","{'Body_Part'}, {'Thing'}, {'Anatomic_Structure...",{human.NCI_C12814}


### Select best direct entry

In [342]:
def select_best_direct_entity(
    src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr
) -> list[OntologyEntryAttr, OntologyEntryAttr]:
    """If there are multiple direct parents, select one and find child element for it."""
    src_parents = next(iter(src_entity.get_direct_parents()))
    tgt_parents = next(iter(tgt_entity.get_direct_parents()))

    src_children = next(iter(src_parents.get_direct_children()))
    tgt_children = next(iter(tgt_parents.get_direct_children()))

    return [x.get_preffered_names() for x in [src_parents, tgt_parents, src_children, tgt_children]]

<b><i> Example ontology</i></b>:

<b> Number </b>: 315

<b> Column name </b>: http://mouse.owl#MA_0000293

<b> All entity names </b>: {'pelvic girdle bone'}

<b> Parents preffered names </b>: "[{'pelvis bone'}, {'trunk bone'}, {'bone'}, {'abdomen/pelvis/perineum bone'}, {'girdle bone'}, {owl.Thing}]"

<b> Direct parents </b>: "{mouse.MA_0000291, mouse.MA_0000532}"

In [ ]:
mouse_example = OntologyEntryAttr("http://mouse.owl#MA_0000293", onto_src)

parent = next(iter(mouse_example.get_direct_parents()))
child = next(iter(parent.get_direct_children()))

mouse_example, parent, child

(mouse.MA_0000293, mouse.MA_0000532, mouse.MA_0000313)

### Format hierarchy

In [371]:
def format_hierarchy(hierarchy_dict:dict[int, set[OntologyEntryAttr]], no_level=False) -> str:
    formatted = []
    for level, parents in sorted(hierarchy_dict.items()):
        parent_name =  next(iter(next(iter(parents)).get_all_entity_names()))
        if no_level:
            formatted.append(parent_name)
        else:
            formatted.append(f"Level {level}: {parent_name}")
    
    if no_level:
        return formatted
    return "\n".join(formatted)

### Prompt templates

In [374]:
def prompt_direct_entity(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Regular prompt that uses natural language and is more intuitive."""
    src_parent, tgt_parent, src_child, tgt_child = select_best_direct_entity(src_entity, tgt_entity)
    return f"""
We have two entities from different biomedical ontologies.  
The first one is "{src_child}", which falls under the category "{src_parent}".  
The second one is "{tgt_child}", which falls under the category "{tgt_parent}".  

Do they mean the same thing? Respond with "True" or "False".
"""


def prompt_direct_entity_ontological(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Ontological prompt that uses ontology-focused language, emphasizing terms like ontological concept,
    hierarchical placement, semantic equivalence, etc.
    """
    src_parent, tgt_parent, src_child, tgt_child = select_best_direct_entity(src_entity, tgt_entity)
    return f"""
Analyze the following entities, each originating from a distinct biomedical ontology. Your task is to assess
whether they represent the **same ontological concept**, considering both their semantic meaning and hierarchical
position.

1. Source entity: "{src_child}"
    - Direct ontological parent: "{src_parent}"

2. Target entity: "{tgt_child}"
    - Direct ontological parent: "{tgt_parent}"

Are these entities **ontologically equivalent** within their respective ontologies? Respond with "True" or "False".
"""


def prompt_sequential_hierarchy(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Regular prompt that uses natural language and is more intuitive."""
    src_hierarchy = format_hierarchy(src_entity.get_parents_by_levels(max_level=3), True)
    tgt_hierarchy = format_hierarchy(tgt_entity.get_parents_by_levels(max_level=3), True)

    return f"""
We have two entities from different biomedical ontologies.  
The first one is "{src_hierarchy[0]}", which belongs to the broader category "{src_hierarchy[1]}" under the even broader category "{src_hierarchy[2]}".  
The second one is "{tgt_hierarchy[0]}", which belongs to the broader category "{tgt_hierarchy[1]}" under the even broader category "{tgt_hierarchy[2]}".  

Do they mean the same thing? Respond with "True" or "False".
"""


def prompt_sequential_hierarchy_ontological(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Ontological prompt that uses ontology-focused language, emphasizing terms like ontological concept,
    hierarchical placement, semantic equivalence, etc.
    """
    src_hierarchy = format_hierarchy(src_entity.get_parents_by_levels(max_level=3))
    tgt_hierarchy = format_hierarchy(tgt_entity.get_parents_by_levels(max_level=3))

    return f"""
The following entities come from distinct biomedical ontologies. Each is represented by its **ontological lineage**, capturing its hierarchical placement from the most general to the most specific level.

1. Source entity ontological lineage:
{src_hierarchy}

2. Target entity ontological lineage:
{tgt_hierarchy}

Based on their **ontological positioning, hierarchical relationships, and semantic alignment**, do these entities represent the **same ontological concept**? Respond with "True" or "False".
"""  # noqa: E501


In [375]:
src_entity, tgt_entity = (
    OntologyEntryAttr("http://mouse.owl#MA_0000293", onto_src),
    OntologyEntryAttr("http://human.owl#NCI_C33075", onto_tgt),
)

print(prompt_sequential_hierarchy(src_entity, tgt_entity))


We have two entities from different biomedical ontologies.  
The first one is "pelvic girdle bone", which belongs to the broader category "girdle bone" under the even broader category "abdomen/pelvis/perineum bone".  
The second one is "Meibomian_Gland", which belongs to the broader category "Sebaceous Gland" under the even broader category "A gland from which secretions reach a free surface of the body by ducts.".  

Do they mean the same thing? Respond with "True" or "False".



### Run experiments

In [ ]:
MAX_WORKERS = 6
MODEL = "gpt-4o-mini"

EXP_NAMES_TO_PROMPT_FUNCTIONS = {
    "direct_entity": prompt_direct_entity,
    "direct_entity_ontological": prompt_direct_entity_ontological,
    "sequential_hierarchy": prompt_sequential_hierarchy,
    "sequential_hierarchy_ontological": prompt_sequential_hierarchy_ontological,
}


oracle_candidate_pairs_path = format_oracle_pairs_filepath(DATASET_NAME, SET_NAME)

oracle_candidate_pairs = read_oracle_pairs(oracle_candidate_pairs_path)[:10]

run_path = format_run_path()

for current_exp_name, prompt_function in EXP_NAMES_TO_PROMPT_FUNCTIONS.items():
    results, tokens_usage, confidences = parallel_samples_process(
        oracle_candidate_pairs, llm_oracle, onto_src, onto_tgt, MODEL, MAX_WORKERS, prompt_function
    )
    prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
        run_path, SET_NAME, MODEL, current_exp_name
    )
    save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])

    plot_usage_histograms(tokens_usage, confidences, do_plot=True, do_print=True, suptitle=current_exp_name)

    predictions = get_predictions_with_gt(run_path, DATASET_NAME, SET_NAME, MODEL, current_exp_name)
    analyze_results(
        predictions,
        print_results=True,
        plot_confusion_matrix=True,
        subtitle=f"{SET_NAME}: {MODEL} {current_exp_name} | ",
        cm_save_path=diagram_path,
        stats_path=stats_path,
    )
    print("\n\n\n")
    break
